# Init

In [12]:
import torch
torch.tensor([
    [10, 11, 12],
    [20, 21, 22],
    [30, 31, 32],
    [30, 31, 32],
    [30, 31, 32],
]).sum(dim=1)

tensor([33, 63, 93, 93, 93])

In [8]:
m

tensor([33, 63, 93])

In [ ]:
module load singularity

In [ ]:
singularity build \
/data/drdcad/sunggu/.singularity_images/cuda-12.2.2-cudnn8-devel-ubuntu22.04.sif \
/data/drdcad/sunggu/.singularity_images/Dockerfile.def

In [ ]:
singularity shell --nv \
  -B /vf,/data,/spin1 \
  -B /data/drdcad/sunggu:/workspace \
  /data/drdcad/sunggu/.singularity_images/cuda-12.2.2-cudnn8-devel-ubuntu22.04.sif

# Install

In [ ]:
# check the spaces
du -h --max-depth=1 /home/kyungs2 2>/dev/null | sort -h

# remove cache
rm -rf /home/kyungs2/.cache
rm -rf /home/kyungs2/.singularity/cache

In [ ]:
# rm -rf .venv

uv venv --python 3.13

uv sync # REMOVE "flash-attn>=2.8.3" from pyproject.toml

uv add flash-attn==2.8.3 --no-build-isolation

source .venv/bin/activate

In [ ]:
git clone https://github.com/YalaLab/rave ../rad-vision-engine
uv pip install -e /data/drdcad/sunggu/0.Challenge/rad-vision-engine

# Data generation for CSV

In [ ]:
# /data/MIDS/datasets/CT-RATE


In [ ]:
# import shutil 

# shutil.copyfile("/data/MIDS/datasets/CT-RATE/dataset/radiology_text_reports/train_reports.csv", "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/train_reports.csv")
# shutil.copyfile("/data/MIDS/datasets/CT-RATE/dataset/radiology_text_reports/validation_reports.csv", "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/validation_reports.csv")



'/data/drdcad/sunggu/0.Challenge/pillar-pretrain/validation_reports.csv'

## csv 만들기

In [310]:
import pandas as pd

train_abn_df =  pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-finetune/train_predicted_labels.csv")
valid_abn_df =  pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-finetune/valid_predicted_labels.csv")

train_rep_df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/train_reports.csv")
valid_rep_df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/validation_reports.csv")



In [311]:
# train_rep_df['Findings_EN']가 NaN인 index 확인
mask = train_rep_df["Findings_EN"].isna()
train_rep_df.loc[mask, "Findings_EN"] = train_rep_df.loc[mask, "Impressions_EN"]
train_rep_df.loc[mask, "Impressions_EN"] = "Not given."

# # train_rep_df['Impressions_EN']가 NaN인 index 확인
mask = train_rep_df["Impressions_EN"].isna()
train_rep_df.loc[mask, "Impressions_EN"] = "Not given."

In [312]:
train_rep_df['final_report'] = "Findings: " + train_rep_df['Findings_EN'].str.strip() + "\nImpression: " + train_rep_df['Impressions_EN'].str.strip()
valid_rep_df['final_report'] = "Findings: " + valid_rep_df['Findings_EN'].str.strip() + "\nImpression: " + valid_rep_df['Impressions_EN'].str.strip()

In [313]:
# train_rep_df와 train_abn_df의 각 "VolumeName"이 일치하는 행을 concat
merged_train_df = pd.merge(train_rep_df, train_abn_df, on='VolumeName', how='inner')
merged_valid_df = pd.merge(valid_rep_df, valid_abn_df, on='VolumeName', how='inner')

In [314]:
merged_train_df['accession'] = merged_train_df['VolumeName'].str.replace('.nii.gz', '', regex=False)
merged_valid_df['accession'] = merged_valid_df['VolumeName'].str.replace('.nii.gz', '', regex=False)

In [315]:
def train_get_series_path(volume_name):
    parts = volume_name.split("_")
    name = parts[0]
    number = parts[1]
    alphabet = parts[2]
    number2_with_nii = parts[3]
    
    series_path = "/data/MIDS/datasets/CT-RATE/dataset/train_fixed/" + name+"_"+number + "/" + name+"_"+number+"_"+alphabet + "/" + name+"_"+number+"_"+alphabet+"_"+number2_with_nii
    return series_path

def valid_get_series_path(volume_name):
    parts = volume_name.split("_")
    name = parts[0]
    number = parts[1]
    alphabet = parts[2]
    number2_with_nii = parts[3]
    
    series_path = "/data/MIDS/datasets/CT-RATE/dataset/valid_fixed/" + name+"_"+number + "/" + name+"_"+number+"_"+alphabet + "/" + name+"_"+number+"_"+alphabet+"_"+number2_with_nii
    return series_path

In [316]:
merged_train_df['series_path'] = merged_train_df['VolumeName'].apply(train_get_series_path)
merged_valid_df['series_path'] = merged_valid_df['VolumeName'].apply(valid_get_series_path)

In [318]:
# chest 아닌거 제외
with open("/data/MIDS/datasets/CT-RATE/dataset/metadata/no_chest_train.txt", "r") as f:
    train_exclude_set = set(line.strip() for line in f if line.strip())
train_exclude_set = list(map(lambda x: x.split("/")[-1].replace(".nii.gz", ""), train_exclude_set))
train_filtered_df = merged_train_df[~merged_train_df['accession'].isin(train_exclude_set)]

with open("/data/MIDS/datasets/CT-RATE/dataset/metadata/no_chest_valid.txt", "r") as f:
    valid_exclude_set = set(line.strip() for line in f if line.strip())
valid_exclude_set = list(map(lambda x: x.split("/")[-1].replace(".nii.gz", ""), valid_exclude_set))
valid_filtered_df = merged_valid_df[~merged_valid_df['accession'].isin(valid_exclude_set)]


In [321]:
# merged_train_df와 merged_valid_df 저장
train_filtered_df.to_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.csv", index=False)
valid_filtered_df.to_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.csv", index=False)

## json 파일 만들기

In [2]:
# "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train.json" 읽기
import pandas as pd

merged_train_df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.csv")
merged_valid_df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.csv")

In [3]:
len(merged_train_df), len(merged_valid_df)

(46397, 3002)

In [4]:
merged_train_df = merged_train_df.rename(columns={"accession": "sample_name", "final_report": "report_metadata"})
merged_valid_df = merged_valid_df.rename(columns={"accession": "sample_name", "final_report": "report_metadata"})

In [5]:
merged_train_df = merged_train_df[["sample_name", "report_metadata"]]
merged_valid_df = merged_valid_df[["sample_name", "report_metadata"]]

In [6]:
len(merged_train_df), len(merged_valid_df)

(46397, 3002)

In [349]:
merged_train_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

merged_valid_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

In [351]:
len(merged_train_df)

46397

# Add Key Phrases

In [25]:
# merged_train_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/train_sentence_labels_final.json", lines=False)
# merged_valid_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/val_sentence_labels_final.json", lines=False)

merged_train_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/train_sentence_labels_with_probs.json", lines=False)
merged_valid_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/val_sentence_labels_with_probs.json", lines=False)

In [26]:
merged_train_df['subject_id'] = merged_train_df['sample_name'].map(lambda x: x.rsplit("_", 1)[0])
merged_valid_df['subject_id'] = merged_valid_df['sample_name'].map(lambda x: x.rsplit("_", 1)[0])


In [27]:
cols = ["sentences", "labels", "volume_label", "sentence_probs"]

lookup = merged_train_json.T[cols].copy()
lookup.index = lookup.index.astype(str)
merged_train_df["subject_id"] = merged_train_df["subject_id"].astype(str)
merged_train_df = merged_train_df.drop(columns=cols + ["dict"], errors="ignore")
merged_train_df = merged_train_df.join(lookup, on="subject_id")

lookup = merged_valid_json.T[cols].copy()
lookup.index = lookup.index.astype(str)
merged_valid_df["subject_id"] = merged_valid_df["subject_id"].astype(str)
merged_valid_df = merged_valid_df.drop(columns=cols + ["dict"], errors="ignore")
merged_valid_df = merged_valid_df.join(lookup, on="subject_id")

In [28]:
label_columns = [
    "Medical material",
    "Arterial wall calcification",
    "Cardiomegaly",
    "Pericardial effusion",
    "Coronary artery wall calcification",
    "Hiatal hernia",
    "Lymphadenopathy",
    "Emphysema",
    "Atelectasis",
    "Lung nodule",
    "Lung opacity",
    "Pulmonary fibrotic sequela",
    "Pleural effusion",
    "Mosaic attenuation pattern",
    "Peribronchial thickening",
    "Consolidation",
    "Bronchiectasis",
    "Interlobular septal thickening",
]

In [29]:
merged_train_df['sentences'].iloc[0]

['Patent trachea and both main bronchi',
 'Linear atelectasis in both lungs',
 'Subsegmental atelectasis in the right middle lobe',
 'Bronchial wall thickening in the bilateral lower lobes',
 'Peribronchial tree-like reticulonodular densities in the bilateral lower lobes',
 'Focal minimal peribronchial consolidation in the lower lobes',
 'Calcific plaques in the aortic arch',
 'Normal mediastinal vascular structures, heart contour, and size',
 'Normal thoracic aorta diameter',
 'No pericardial effusion or thickening',
 'Collapsed left subclavian vein',
 'Normal thoracic esophagus caliber without significant wall thickening',
 'No enlarged lymph nodes in the prevascular, paratracheal, subcarinal, or bilateral hilar-axillary regions',
 'Multiple venous collaterals in the anterior left chest wall associated with the anterior jugular vein at the level of the right sternoclavicular junction',
 'Osteophytes with anterior extension in the thoracic vertebrae',
 'Atrophic left kidney',
 'Right 

In [30]:
merged_train_df['key_phrase'] = merged_train_df['sentences']
merged_valid_df['key_phrase'] = merged_valid_df['sentences']

In [31]:
merged_train_df['key_phrase'].iloc[0]

['Patent trachea and both main bronchi',
 'Linear atelectasis in both lungs',
 'Subsegmental atelectasis in the right middle lobe',
 'Bronchial wall thickening in the bilateral lower lobes',
 'Peribronchial tree-like reticulonodular densities in the bilateral lower lobes',
 'Focal minimal peribronchial consolidation in the lower lobes',
 'Calcific plaques in the aortic arch',
 'Normal mediastinal vascular structures, heart contour, and size',
 'Normal thoracic aorta diameter',
 'No pericardial effusion or thickening',
 'Collapsed left subclavian vein',
 'Normal thoracic esophagus caliber without significant wall thickening',
 'No enlarged lymph nodes in the prevascular, paratracheal, subcarinal, or bilateral hilar-axillary regions',
 'Multiple venous collaterals in the anterior left chest wall associated with the anterior jugular vein at the level of the right sternoclavicular junction',
 'Osteophytes with anterior extension in the thoracic vertebrae',
 'Atrophic left kidney',
 'Right 

In [32]:
merged_train_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

merged_valid_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_prob.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

In [33]:
merged_train_df[merged_train_df['report_metadata'].isna()]


,sample_name,report_metadata,subject_id,key_phrase,sentences,labels,volume_label,sentence_probs


In [ ]:
# text - train
export TOKENIZERS_PARALLELISM=False &&
export OMP_NUM_THREADS=1 && \
export MKL_NUM_THREADS=1 && \
export OPENBLAS_NUM_THREADS=1 && \
uv run python scripts/cache_ctrate_embeddings_phrase_align.py \
--model-name /data/drdcad/sunggu/llm_models/Qwen3-Embedding-8B \
--json-file /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_phrase_align.json \
--cache-dir data/text_cache_qwen3_embedding_8b_phrase/train_2048 \
--batch-size 32 \
--num-processes 1

# text - valid (done)
export TOKENIZERS_PARALLELISM=False &&
export OMP_NUM_THREADS=1 && \
export MKL_NUM_THREADS=1 && \
export OPENBLAS_NUM_THREADS=1 && \
uv run python scripts/cache_ctrate_embeddings_phrase_align.py \
--model-name /data/drdcad/sunggu/llm_models/Qwen3-Embedding-8B \
--json-file /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_phrase_align.json \
--cache-dir data/text_cache_qwen3_embedding_8b_phrase/val_2048 \
--batch-size 32 \
--num-processes 1


In [40]:
import glob
import pandas as pd


train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob.json", lines=True)
valid = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_prob.json", lines=True)
# k = glob.glob("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b_phrase/train/embeddings/*")

In [41]:
train[train['sentences'].isna()]['sample_name'].values

array(['train_84_c_1', 'train_84_c_2', 'train_1364_c_1', 'train_1364_c_2',
       'train_2317_d_1', 'train_2317_d_2', 'train_7778_b_1',
       'train_7778_b_2', 'train_8244_h_1', 'train_8244_h_2',
       'train_10328_a_1', 'train_10328_a_2', 'train_10759_b_1',
       'train_10759_b_2', 'train_10962_b_1', 'train_10962_b_2',
       'train_16550_c_1', 'train_16550_c_2', 'train_17046_b_1',
       'train_17046_b_2', 'train_17597_a_1', 'train_17597_a_2'],
      dtype=object)

In [42]:
valid[valid['sentences'].isna()]

,sample_name,report_metadata,subject_id,key_phrase,sentences,labels,volume_label,sentence_probs


In [43]:
len(train)

46397

In [44]:
drop_cases = ['train_84_c_1', 'train_84_c_2', 'train_1364_c_1', 'train_1364_c_2',
       'train_2317_d_1', 'train_2317_d_2', 'train_7778_b_1',
       'train_7778_b_2', 'train_8244_h_1', 'train_8244_h_2',
       'train_10328_a_1', 'train_10328_a_2', 'train_10759_b_1',
       'train_10759_b_2', 'train_10962_b_1', 'train_10962_b_2',
       'train_16550_c_1', 'train_16550_c_2', 'train_17046_b_1',
       'train_17046_b_2', 'train_17597_a_1', 'train_17597_a_2']

train = train[~train['sample_name'].isin(drop_cases)].reset_index(drop=True)

In [45]:
len(train)

46375

In [46]:
# 다시 저장

train.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

# Vision cache

In [ ]:
mkdir -p data/text_cache_qwen3_embedding_8b


In [ ]:
# # vision - train
# uv run vision-engine process \
# --config /data/drdcad/sunggu/0.Challenge/rad-vision-engine/configs/ct_chest.yaml \
# --input-series-csv /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train.csv \
# --output data/ctrate/ctrate_cache_1.25mm/train \
# --workers 128


# Subset
# uv run vision-engine process \
# --config /data/drdcad/sunggu/0.Challenge/rad-vision-engine/configs/ct_chest.yaml \
# --input-series-csv /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_subset.csv \
# --output data/ctrate/ctrate_cache_1.25mm/train \
# --workers 128

# # vision - valid
# uv run vision-engine process \
# --config /data/drdcad/sunggu/0.Challenge/rad-vision-engine/configs/ct_chest.yaml \
# --input-series-csv /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid.csv \
# --output data/ctrate/ctrate_cache_1.25mm/valid \
# --workers 128


In [ ]:
# 다시 해야 됨.
import pandas as pd

df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train.csv")
# df에서 20000~끝까지 index를 색인
df_subset = df.iloc[45000:]
len(df_subset)
# df_subset 저장

df_subset.to_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_subset.csv", index=False)

# Text cache

In [ ]:
# text - train
# export TOKENIZERS_PARALLELISM=False &&
# export OMP_NUM_THREADS=1 && \
# export MKL_NUM_THREADS=1 && \
# export OPENBLAS_NUM_THREADS=1 && \
# uv run python scripts/cache_merlin_embeddings.py \
# --model-name /data/drdcad/sunggu/llm_models/Qwen3-Embedding-8B \
# --json-file /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.json \
# --cache-dir data/text_cache_qwen3_embedding_8b/train \
# --batch-size 32 \
# --num-processes 1

# text - valid (done)
# export TOKENIZERS_PARALLELISM=False &&
# export OMP_NUM_THREADS=1 && \
# export MKL_NUM_THREADS=1 && \
# export OPENBLAS_NUM_THREADS=1 && \
# uv run python scripts/cache_merlin_embeddings.py \
# --model-name /data/drdcad/sunggu/llm_models/Qwen3-Embedding-8B \
# --json-file /data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.json \
# --cache-dir data/text_cache_qwen3_embedding_8b/valid_test \
# --batch-size 32 \
# --num-processes 8


In [ ]:
# tmux new -s my_work
# 나갈때 Ctrl + B를 누른 뒤 손을 떼고 D키를 누릅니다.
# tmux a -t my_work

# Mapping csv 만들기

In [ ]:
# manifest.csv == mapping.csv 형식이 다름. (아래 참고)

# "sample_name", "image_cache_path"

In [6]:
import pandas as pd

train_df   = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.csv")
mapping_df = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/ctrate/ctrate_cache_1.25mm/train/mapping.csv")

In [8]:
len(mapping_df)

5

In [470]:
mapping_df.head()

,source_path,output_path
0,/data/MIDS/datasets/CT-RATE/dataset/train_fixe...,data/ctrate/ctrate_cache_1.25mm/train/train_84...
1,/data/MIDS/datasets/CT-RATE/dataset/train_fixe...,data/ctrate/ctrate_cache_1.25mm/train/train_12...
2,/data/MIDS/datasets/CT-RATE/dataset/train_fixe...,data/ctrate/ctrate_cache_1.25mm/train/train_12...
3,/data/MIDS/datasets/CT-RATE/dataset/train_fixe...,data/ctrate/ctrate_cache_1.25mm/train/train_12...
4,/data/MIDS/datasets/CT-RATE/dataset/train_fixe...,data/ctrate/ctrate_cache_1.25mm/train/train_15...


In [ ]:
new_mapping_df = pd.DataFrame()

new_mapping_df['source_path'] = train_df['series_path']
new_mapping_df['output_path'] = ("data/ctrate/ctrate_cache_1.25mm/train/"+ df['accession'].astype(str)+ ".1.0")
new_mapping_df['sample_name'] = train_df['accession']
new_mapping_df['image_cache_path'] = new_mapping_df['output_path'].copy()

In [472]:
# new_mapping_df 저장

new_mapping_df.to_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/ctrate/ctrate_cache_1.25mm/train/mapping_path.csv", index=False)

In [478]:
len(new_mapping_df)

46397

# 학습

In [ ]:
# train 학습

bash /data/drdcad/sunggu/0.Challenge/pillar-pretrain/train_unimodal_pillar0_ctrate_chest_ct_1node.sh

In [480]:
# auto_map의 경우 복사해야함.

# from transformers import AutoModel
# import os

# os.environ['HF_HUB_OFFLINE'] = "1"
# os.environ['TRANSFORMERS_OFFLINE'] = "1"
# os.environ['TRANSFORMERS_CACHE'] = "${workspaceFolder}/.hf_cache"
# os.environ['HF_HOME'] = "${workspaceFolder}/.hf_cache" 


# pretrained_model = AutoModel.from_pretrained('/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/multimodal_atlas_d2d2d8_small_r1024_008')

# config.json에 auto_map이 있는 경우 + trust_remote_code=True 일 경우, cache로 복사해주어야 함...
# AutoModel.from_pretrained

import shutil

!rm -rf '/home/kyungs2/.cache/huggingface/modules/transformers_modules/multimodal_atlas_d2d2d8_small_r1024_008'
shutil.copytree('/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/multimodal_atlas_d2d2d8_small_r1024_008', '/home/kyungs2/.cache/huggingface/modules/transformers_modules/multimodal_atlas_d2d2d8_small_r1024_008')

'/home/kyungs2/.cache/huggingface/modules/transformers_modules/multimodal_atlas_d2d2d8_small_r1024_008'

# 디버깅

In [ ]:
train_4677_a_2

In [ ]:
merged_train_filtered_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.json", lines=True)
metadata_train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b/train/metadata.json")
merged_train_filtered = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered.csv")
mapping_train = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/ctrate/ctrate_cache_1.25mm/train/mapping_path.csv")

merged_valid_filtered_json = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.json", lines=True)
metadata_valid = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b/val/metadata.json")
merged_valid_filtered = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered.csv")
mapping_valid = pd.read_csv("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/ctrate/ctrate_cache_1.25mm/val/mapping_path.csv")


In [486]:
len(merged_train_filtered_json), len(metadata_train), len(merged_train_filtered), len(mapping_train)

(46397, 46397, 46397, 46397)

In [487]:
len(merged_valid_filtered), len(metadata_valid), len(merged_valid_filtered), len(mapping_valid)

(3002, 3039, 3002, 3002)

In [413]:
merged_train_filtered_json[merged_train_filtered_json['sample_name'] == "train_4677_a_2"]

,sample_name,report_metadata
10901,train_4677_a_2,Findings: There are findings consistent with e...


In [422]:
merged_train_filtered[merged_train_filtered['accession'] == "train_4677_a_2"]['series_path'].iloc[0]

'/data/MIDS/datasets/CT-RATE/dataset/train_fixed/train_4677/train_4677_a/train_4677_a_2.nii.gz'

In [437]:
metadata[metadata['text_to_path'] == "embeddings/420e642a16f9020b59cd0fc6ac7b8bf3a4df656022b50e8e76fefc8a7095ef16.pt"]

,model_name,normalize_embeddings,max_length,text_to_path,embedding_dim,version,dataset_name,split
420e642a16f9020b59cd0fc6ac7b8bf3a4df656022b50e8e76fefc8a7095ef16,/data/drdcad/sunggu/llm_models/Qwen3-Embedding-8B,True,512,embeddings/420e642a16f9020b59cd0fc6ac7b8bf3a4d...,4096,1,NaN,NaN


In [443]:
len(metadata)

46397

In [444]:
import glob

a = glob.glob("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/ctrate/ctrate_cache_1.25mm/train/train_*")

In [447]:
len(a)

47149

In [448]:
len(mapping_train)

46395

In [438]:
import torch

t = torch.load("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b/train/embeddings/420e642a16f9020b59cd0fc6ac7b8bf3a4df656022b50e8e76fefc8a7095ef16.pt")

In [425]:
import nibabel as nib
import numpy as np

path = "/data/MIDS/datasets/CT-RATE/dataset/train_fixed/train_4677/train_4677_a/train_4677_a_2.nii.gz"

t = nib.load(path)

# array 추출
arr = t.get_fdata(dtype=np.float32)

print(type(arr))
print(arr.shape)
print(arr.dtype)
print(arr.min(), arr.max())

<class 'numpy.ndarray'>
(512, 512, 274)
float32
-1190.0 3488.0


In [389]:
import hashlib

# text_cache_key = str("train_4677_a_2")
# text_cache_key = hashlib.sha256(text_cache_key.encode()).hexdigest()

meta_train_set = set([hashlib.sha256(str(i).encode()).hexdigest() for i in merged_train_filtered['accession']])



In [410]:

text_cache_key = str("train_4677_a_2")
text_cache_key = hashlib.sha256(text_cache_key.encode()).hexdigest()

In [411]:
text_cache_key

'420e642a16f9020b59cd0fc6ac7b8bf3a4df656022b50e8e76fefc8a7095ef16'

In [408]:
meta_train_set - set(metadata.index)

set()

In [409]:
set(metadata.index) -  meta_train_set

set()

In [399]:
metadata.index[0]

'b069b456aa9cfc1aa82661b898331f579336176116dc4d9a3aeac1ea8770d3b4'

In [406]:
len(meta_train_set)

46397

In [407]:
len(metadata)

46397

In [11]:
import shutil

!rm -rf '/home/kyungs2/.cache/huggingface/modules/transformers_modules/multimodal_atlas_d2d2d8_small_r1024_008'
!rm -rf '/home/kyungs2/.cache/huggingface/modules/transformers_modules/Pillar0-ChestCT'
!rm -rf '/home/kyungs2/.cache/huggingface/modules/transformers_modules/Pillar0-AbdomenCT'

shutil.copytree("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/multimodal_atlas_d2d2d8_small_r1024_008", "/home/kyungs2/.cache/huggingface/modules/transformers_modules/multimodal_atlas_d2d2d8_small_r1024_008")
shutil.copytree("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/Pillar0-ChestCT", "/home/kyungs2/.cache/huggingface/modules/transformers_modules/Pillar0-ChestCT")
shutil.copytree("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/Pillar0-AbdomenCT", "/home/kyungs2/.cache/huggingface/modules/transformers_modules/Pillar0-AbdomenCT")

'/home/kyungs2/.cache/huggingface/modules/transformers_modules/Pillar0-AbdomenCT'

In [9]:
cd /home/kyungs2/.cache/huggingface/modules/transformers_modules

/vf/users/drdcad/sunggu/.cache/huggingface/modules/transformers_modules


In [10]:
ls

__init__.py                               Pillar0-AbdomenCT/  __pycache__/
multimodal_atlas_d2d2d8_small_r1024_008/  Pillar0-ChestCT/    YalaLab/


In [388]:
meta_train_set

{'...',
 ...}

In [8]:
import SimpleITK as sitk
# path = "/data/MIDS/datasets/CT-RATE/dataset/train_fixed/train_4677/train_4677_a/train_4677_a_2.nii.gz"
path = "/data/MIDS/datasets/CT-RATE/dataset/train_fixed/train_40/train_40_a/train_40_a_1.nii.gz"
img = sitk.ReadImage(path)
arr = sitk.GetArrayFromImage(img)

print("array.shape:", arr.shape)
# print("sitk.GetSize():", img.GetSize())      # (X, Y, Z)
# print("reversed GetSize():", img.GetSize()[::-1])  # equals arr.shape
# print("spacing:", img.GetSpacing())
# print("direction:", img.GetDirection())
# print("origin:", img.GetOrigin())

array.shape: (252, 512, 512)


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, LlamaModel

class JEPA_Predictor(nn.Module):
    def __init__(
        self,
        llama_model_name: str = "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/Llama-3.2-1B",
        vision_input_dim: int = 384,
        embed_dim: int = 1152,
        start_layer: int = 8,
        end_layer: int = 16,
        max_query_tokens: int = 512,
    ):
        super().__init__()
        self.max_query_tokens = max_query_tokens
        self.embed_dim = embed_dim

        # ── Tokenizer ───────────────────────
        self.tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # ── Load Model ───────────────────────
        full_model = LlamaModel.from_pretrained(llama_model_name)
        llama_hidden_size = full_model.config.hidden_size
        
        self.embed_tokens = full_model.embed_tokens
        self.rotary_emb   = full_model.rotary_emb 
        self.layers = nn.ModuleList(full_model.layers[start_layer:end_layer])
        self.norm = full_model.norm
        del full_model

        # Causal Flag 비활성화 (양방향 어텐션 유도)
        for layer in self.layers:
            if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "is_causal"):
                layer.self_attn.is_causal = False

        # ── Linear projections ─────────────────────────────────────────
        self.vision_proj = nn.Linear(vision_input_dim, llama_hidden_size)
        self.output_proj = nn.Linear(llama_hidden_size, embed_dim)

    def _tokenize(self, queries: list[str], device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:        
        enc = self.tokenizer(
            queries,
            padding="max_length",
            truncation=True,
            max_length=self.max_query_tokens,
            return_tensors="pt",
        )
        return enc["input_ids"].to(device), enc["attention_mask"].to(device)

    def forward(self, vision_embeds: torch.Tensor, queries: list[str]) -> torch.Tensor:
        device = vision_embeds.device
        B = vision_embeds.size(0)
        assert len(queries) == B, "queries 길이가 배치 크기와 다릅니다."

        # ── 1. Query 텍스트 처리 ───────────────
        input_ids, attention_mask = self._tokenize(queries, device)
        N_q = input_ids.size(1)
        query_embeds = self.embed_tokens(input_ids)

        # ── 3. Vision 임베딩 Projection ──────
        vis_embeds = self.vision_proj(vision_embeds)
        N_v = vis_embeds.size(1)

        # ── 4. Concat: [visual tokens | query tokens] ──────────────────
        seq = torch.cat([vis_embeds, query_embeds], dim=1)
        seq_length = seq.size(1)

        # ── 5. Attention mask 구성 (causal mask 없음, 양방향) ──────────────────
        vis_mask = torch.ones(B, N_v, device=device, dtype=attention_mask.dtype)
        full_mask = torch.cat([vis_mask, attention_mask], dim=1)  # (B, N_v+N_q)

        # [수정] dtype mismatch를 방지하기 위해 fp16/bf16에 안전한 최솟값 사용
        dtype_min = torch.finfo(seq.dtype).min
        ext_mask = (1.0 - full_mask[:, None, None, :].to(seq.dtype)) * dtype_min

        # ── 5.5. Position IDs 및 RoPE 임베딩 계산 ───────────────
        position_ids = torch.arange(0, seq_length, dtype=torch.long, device=device).unsqueeze(0).expand(B, -1)
        position_embeddings = self.rotary_emb(seq, position_ids)
        
        # ── 6. Transformer forward ─────────────────────────────────────
        hidden = seq
        for layer in self.layers:
            layer_out = layer(
                hidden,
                attention_mask=ext_mask,
                position_embeddings=position_embeddings, 
                use_cache=False,
                output_attentions=False,
            )
            hidden = layer_out[0]
        
        hidden = self.norm(hidden)

        # ── 7. Average Pooling ───────────────────────
        full_pool_mask = full_mask.float().unsqueeze(-1)  # (B, N_v+N_q, 1)
        
        summed = (hidden * full_pool_mask).sum(dim=1)
        count  = full_pool_mask.sum(dim=1).clamp(min=1)
        pooled = summed / count

        # ── 8. Output projection ───────────────
        s_hat_y = self.output_proj(pooled)
        return s_hat_y


In [3]:
# 가상 환경 내 nvidia 패키지 경로를 LD_LIBRARY_PATH에 추가
export LD_LIBRARY_PATH="$VIRTUAL_ENV/lib/python3.13/site-packages/nvidia/cuda_nvrtc/lib:$LD_LIBRARY_PATH"


SyntaxError: invalid syntax (1540189172.py, line 2)

In [1]:
import numpy as np
import torch
from torchcodec.decoders import VideoDecoder
from transformers import AutoModelForVideoClassification, AutoVideoProcessor


processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
model = AutoModel.from_pretrained(
    "facebook/vjepa2-vitl-fpc64-256",
    device_map="auto",
    attn_implementation="sdpa"
)

video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/archery/-Qz25rXdMjE_000014_000024.mp4"

vr = VideoDecoder(video_url)
frame_idx = np.arange(0, 64) # choosing some frames. here, you can define more complex sampling strategy
video = vr.get_frames_at(indices=frame_idx).data  # T x C x H x W
video = processor(video, return_tensors="pt").to(model.device)
outputs = model(**video)

# V-JEPA 2 encoder outputs, same as calling `model.get_vision_features()`
encoder_outputs = outputs.last_hidden_state

# V-JEPA 2 predictor outputs
predictor_outputs = outputs.predictor_output.last_hidden_state

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.7.1+cu126) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call

In [1]:
import numpy as np
import torch
import urllib.request
import torchvision.io
from transformers import AutoVideoProcessor, AutoModel

# 1. 모델 및 프로세서 로드
processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
model = AutoModel.from_pretrained(
    "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/checkpoints/vjepa2-vitl-fpc64-256",
    device_map="auto",
    attn_implementation="sdpa"
)

# 2. 비디오 다운로드 (torchcodec을 쓰지 않기 위해 로컬 파일로 저장)
video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/archery/-Qz25rXdMjE_000014_000024.mp4"
video_path = "temp_video.mp4"
urllib.request.urlretrieve(video_url, video_path)

# 3. torchvision으로 비디오 읽기
# video_frames 형태: [T, H, W, C] (텐서)
video_frames, _, info = torchvision.io.read_video(video_path, pts_unit='sec')

# 4. 프레임 샘플링
# 비디오의 전체 길이 내에서 최대 64프레임 추출
max_frames = min(64, len(video_frames))
frame_idx = np.arange(0, max_frames)
video = video_frames[frame_idx]  # T x H x W x C

# 5. 채널 순서 변경 (T x C x H x W) 및 numpy 변환
video = video.permute(0, 3, 1, 2).numpy()

# 6. 프로세서 및 모델 통과
video_inputs = processor(video, return_tensors="pt").to(model.device)
outputs = model(**video_inputs)

# 7. V-JEPA 2 결과 확인
# Encoder 출력 (Classification, Video QA 등에 활용)
encoder_outputs = outputs.last_hidden_state

# Predictor 출력 (Action Anticipation 등 미래 예측에 활용)
predictor_outputs = outputs.predictor_output.last_hidden_state

print("Encoder Output Shape:", encoder_outputs.shape)
print("Predictor Output Shape:", predictor_outputs.shape)


/vf/users/drdcad/sunggu/0.Challenge/pillar-pretrain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/vf/users/drdcad/sunggu/0.Challenge/pillar-pretrain/.venv/lib/python3.13/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Encoder Output Shape: torch.Size([1, 8192, 1024])
Predictor Output Shape: torch.Size([1, 8192, 1024])


In [4]:
video_inputs['pixel_values_videos'].shape

torch.Size([1, 64, 3, 256, 256])

In [6]:
(64//2)*(256//16)*(256//16)

8192

In [5]:
video_inputs

{'pixel_values_videos': tensor([[[[[-0.6281, -0.5253, -0.4054,  ..., -1.1589, -1.0904, -0.9363],
           [-0.7650, -0.6623, -0.5424,  ..., -1.1589, -1.1247, -1.0219],
           [-0.7993, -0.7479, -0.6965,  ..., -1.1589, -1.1418, -1.0562],
           ...,
           [ 2.2147,  2.2147,  2.2147,  ...,  2.1462,  2.1290,  2.1119],
           [ 2.2318,  2.2318,  2.2318,  ...,  2.0777,  2.0777,  2.0948],
           [ 2.2318,  2.2318,  2.2489,  ...,  1.9920,  2.0605,  2.0948]],

          [[-0.0924,  0.0126,  0.1352,  ..., -0.6877, -0.6176, -0.4601],
           [-0.2325, -0.1275, -0.0224,  ..., -0.6877, -0.6527, -0.5476],
           [-0.2850, -0.2325, -0.1800,  ..., -0.6877, -0.6702, -0.5826],
           ...,
           [ 2.3585,  2.3585,  2.3585,  ...,  2.3761,  2.3585,  2.3410],
           [ 2.3761,  2.3761,  2.3761,  ...,  2.3235,  2.3235,  2.3235],
           [ 2.3936,  2.3761,  2.3936,  ...,  2.2360,  2.3060,  2.3410]],

          [[-0.7587, -0.6541, -0.5321,  ..., -1.0550, -0.9853, -

# check PhraseAlign sentences

In [4]:
# "/data/drdcad/sunggu/0.Challenge/pillar-finetune/train_sentence_labels.json" json 읽기

import json

with open("/data/drdcad/sunggu/0.Challenge/pillar-finetune/train_sentence_labels.json", "r") as f:
    train_data = json.load(f)

with open("/data/drdcad/sunggu/0.Challenge/pillar-finetune/val_sentence_labels.json", "r") as f:
    val_data = json.load(f)


In [5]:
len(train_data), len(val_data)

(23850, 1551)

In [15]:
train_data.keys()

dict_keys(['train_10000_a', 'train_10001_a', 'train_10002_a', 'train_10003_a', 'train_10004_a', 'train_10005_a', 'train_10006_a', 'train_10006_b', 'train_10007_a', 'train_10008_a', 'train_10009_a', 'train_1000_a', 'train_10010_a', 'train_10011_a', 'train_10012_a', 'train_10013_a', 'train_10013_b', 'train_10014_a', 'train_10015_a', 'train_10016_a', 'train_10016_b', 'train_10016_c', 'train_10016_d', 'train_10017_a', 'train_10018_a', 'train_10019_a', 'train_1001_a', 'train_1001_b', 'train_10020_a', 'train_10021_a', 'train_10022_a', 'train_10023_a', 'train_10024_a', 'train_10025_a', 'train_10026_a', 'train_10027_a', 'train_10027_b', 'train_10028_a', 'train_10029_a', 'train_1002_a', 'train_1002_b', 'train_10030_a', 'train_10030_b', 'train_10031_a', 'train_10032_a', 'train_10033_a', 'train_10034_a', 'train_10035_a', 'train_10036_a', 'train_10037_a', 'train_10038_a', 'train_10039_a', 'train_1003_a', 'train_10040_a', 'train_10041_a', 'train_10042_a', 'train_10043_a', 'train_10043_b', 'train_10

In [16]:
train_data['train_10000_a'].keys()

dict_keys(['sentences', 'labels', 'volume_label'])

In [17]:
train_data['train_10000_a']

{'sentences': ['Patent trachea and both main bronchi',
  'Indeterminate subcentimeter nodules in both lungs',
  'No masses or infiltrative lesions in either lung',
  'No pleural effusion',
  'Normal heart contour and size',
  'No pericardial effusion',
  'Normal widths of the mediastinal main vascular structures',
  'Mediastinal structures cannot be optimally evaluated due to lack of intravenous contrast',
  'No pathologically enlarged lymph nodes in the mediastinum or hilar regions',
  'Normal esophagus caliber without pathological wall thickening',
  'Normal thoracic vertebral body heights, alignment, and densities',
  'Preserved intervertebral disc heights',
  'Patent neural foramina',
  'No lytic or destructive bone lesions',
  'No free fluid or collections in the upper abdomen',
  'No pathologically enlarged lymph nodes in the upper abdomen',
  'No masses with distinguishable borders in the upper abdominal organs'],
 'labels': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [47]:
label_columns = [
    "Medical material",
    "Arterial wall calcification",
    "Cardiomegaly",
    "Pericardial effusion",
    "Coronary artery wall calcification",
    "Hiatal hernia",
    "Lymphadenopathy",
    "Emphysema",
    "Atelectasis",
    "Lung nodule",
    "Lung opacity",
    "Pulmonary fibrotic sequela",
    "Pleural effusion",
    "Mosaic attenuation pattern",
    "Peribronchial thickening",
    "Consolidation",
    "Bronchiectasis",
    "Interlobular septal thickening",
]

import pandas as pd


train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob.json", lines=True)
val   = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_prob.json", lines=True)

# none 제거
train_clean = train[train['labels'].notna()]
val_clean   = val[val['labels'].notna()]

In [49]:
import numpy as np

all_list = []
for i in train_clean['labels'].values:
    try:
        all_list += np.sum(np.array(i), axis=1).tolist()
    except Exception as e:
        print(f"Error: {e}")
        print(f"shape: {np.array(i).shape}, value: {i}")

In [50]:
all_list = []
for i in val_clean['labels'].values:
    try:
        all_list += np.sum(np.array(i), axis=1).tolist()
    except Exception as e:
        print(f"Error: {e}")
        print(f"shape: {np.array(i).shape}, value: {i}")

In [51]:
cnt=0
for i in all_list:
    if i >= 2:
        cnt+=1

In [52]:
cnt

1817

# Text 길이 체크

In [ ]:
import pandas as pd

train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_phrase_align.json", lines=True)
val   = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_phrase_align.json", lines=True)


In [76]:
train["len"] = list(map(lambda x: len(x), train['report_metadata']))

In [ ]:
len(self.tokenizer(
            ["Findings: Mediastinal structures cannot be evaluated optimally because contrast material is not given. As far as can be followed: It was learned from the patient's history that he had undergone left upper lobectomy. Total loss of aeration is observed in the lower lobe of the left lung, and there is an appearance of consolidation-soft tissue density in the left pulmonary hilus, whose borders cannot be clearly distinguished. When the previous examinations of the patient are examined, increased FDG uptake is observed in PET-CT in this localization. In addition, CT examination reveals that this appearance narrows the pulmonary artery. When evaluated together with these findings, the described appearance was thought to be a mass. It is observed that the described lesion extends along the mediastinal pleura, especially in the medial and anterior sections, and there are pleural thickenings in the left hemithorax. Pleural effusion is observed on the left. Hyperdense appearances are observed in the effusion and it is thought to be due to pleurodesis. In addition, there are thin septum-like appearances in places within the effusion. Minimal pleural effusion is also observed on the right, and there is a drainage catheter in the effusion at the level of the lower lobe of the lung. A mass whose borders cannot be distinguished from the pleura is observed in the left hemithorax at the level where the 2nd and 3rd ribs articulate with the sternum. The anterior-posterior diameter of the described mass was approximately 35 mm at its thickest point. Just caudal to this, another soft tissue lesion measuring approximately 20 mm in diameter is observed in the subcutaneous adipose tissue, adjacent to the 5th rib and sternum joint, and it was evaluated in favor of metastasis. There is a mass with the longest diameter of 54 mm in the subcutaneous adipose tissue in the lateral part of the left hemithorax. In addition, there is another similar solid mass with a diameter of 54 mm on the posterolateral at the lower part of the left hemithorax. Those masses were evaluated in favor of metastases. In the left hemithorax, at the level of the middle part of the hemithorax, infiltrative appearances are observed in the intercostal area and subcutaneous adipose tissue, in soft tissue density whose borders cannot be distinguished from the muscle groups and pleura in this localization. Although these appearances are nonspecific in this examination, when evaluated together with the patient's previous examinations, these appearances were thought to be neoplastic infiltration. Heart contour and size are normal. There is minimal pericardial effusion. The widths of the mediastinal main vascular structures are normal. There are lymphadenopathies in the mediastinum and hilar regions. In addition, nodular lesions are observed in the paracardiac fat pad and were evaluated in favor of lymphadenopathies. The largest of the described lymphadenopathies is observed in the right hilar region and its short diameter is 22 mm. There is no pathological wall thickness increase in the esophagus within the sections. There is no obstructive pathology in the trachea and right main bronchus. There are emphysematous changes and occasional pleuroparenchymal sequelae in the right lung. In the apical segment of the upper lobe of the right lung, an irregularly bordered soft tissue density with a longest diameter of 30 mm is observed. When this appearance was evaluated together with previous examinations, it was thought to be a metastatic mass. The mass boundaries cannot be distinguished from the pleura. In addition, there are masses in the right lung adjacent to the upper and lower lobes, which are thought to originate primarily from the pleura and most of them are in the form of plaques. The largest of these masses are observed within the horizontal and oblique fissures and measure approximately 29 mm in the thickest part (series 2 section 173). These appearances were evaluated in favor of metastases. There are also millimetric nodules in the right lung. No infiltrative lesion was detected in the right lung. Hypodense lesions are observed in the liver. These lesions can be observed in both lobes. When evaluated together with previous examinations, it was understood that these appearances were metastases. The larger metastatic lesions are observed in segment 4A and segment 3, and their longest diameters were 48 mm and 41 mm at their widest points, respectively. There is a nodular lesion on the lateral leg of the left adrenal gland, measuring approximately 20 mm in longest diameter. Again, another nodular lesion measuring approximately 7 mm in diameter is observed in the vicinity of the left adrenal gland corpus. The appearances described in the presence of primary disease were primarily evaluated in favor of metastases. Thickening is also observed in the medial leg of the right adrenal gland. The described appearance is also present in the previous examination of the patient. However, it was found to be enlarged on this examination. Therefore, it was thought to be metastasis. No upper abdominal free fluid was detected in the sections. There are lymphadenopathies in the interaorthocaval and paracaval regions. The largest of the described lymphadenopathies is observed in the paracaval area and its short diameter is 15 mm. No appearance that can be evaluated in favor of metastasis was detected in the bone structures within the sections. There was no significant difference in the size of lymphadenopathy in the mediastinum and in the hilar region. No significant difference was found in the size of the mass in the infiltrative character described in the left hemithorax. It is understood that the masses observed in the subcutaneous adipose tissue in the left hemithorax have grown significantly. No significant difference was found in the sizes of the masses with infiltrative character described in the intercostal region. It is understood that the dimensions of the mass described in the upper lobe apical segment of the right lung have increased slightly. In addition, there is a slight increase in the size of the masses described in the right lung and thought to be of pleural origin. There is an increase in the size of metastatic lesions observed in the liver. An increase in the size of both masses adjacent to the left adrenal gland was observed. There is also an increase in lesion size in the right adrenal gland.\nImpression: Operated lung Ca, left upper lobectomy, total aeration loss in the left lung and infiltrative appearance in the left pulmonary hilum, which is thought to be a mass when evaluated together with the patient's previous examinations, masses compatible with metastasis-infiltration in the subcutaneous fat tissue and muscle groups in the left hemithorax, metastatic in the right lung masses, lymphadenopathies in the mediastinal and hilar region, liver metastases, metastases in both adrenal glands, metastases adjacent to the left adrenal gland, intraabdominal lymphadenopathies."],
            padding=False,
            truncation=False,
            return_tensors=None
        )['input_ids'][0]) # 1479

In [ ]:
len(self.tokenizer(
            ['Findings: There is a mass lesion originating from the right main bronchus in the right lung hilum, invading the mediastinum superiorly, adjacent to the right trachea, and infiltratively extending to the right anterior scalene muscles. Invasion of the anterior scalene muscle and brachial plexus trunks is observed in the supraclavicular fossa on the right. The first rib extends to the vertebral junction. It extends anteriorly to the strep muscles in the right paratracheal area. Its borders with the right thyroid lobe anteriorly and the esophagus posteromedially cannot be distinguished. It is doubtful in favor of invasion. Since contrast material was not given, its relation with vascular structures was not evaluated. It extends to the left CCA posterior in the previous examination, and it extends to a similar localization in the current examination. Inferiorly, it continues in the mediastinum until distal to the right intermediate bronchus. Pathological LAP with calcification of approximately 2 cm in the subcarinal area was also present in the current examination, and no significant difference was detected. In the evaluation of the lung parenchyma window; The right lung upper lobe bronchus is narrowed by the mass lesion before the bifurcation. It cannot be followed distal to the bifurcation, it is obstructed by the apical extension of the primary mass. Pleural effusion is observed adjacent to the mass in the apical segment of the upper lobe of the right lung. Since no contrast material is given, its dimensions cannot be distinguished from the mass. Traction bronchiectasis is observed in the right lung upper lobe posterior segment bronchi. More prominent bronchiectasis and bronchial wall thickness increases are observed in the basal segment bronchi in both lungs. Intrapulmonary metastases measuring 12 mm in the anterobasal segment of the lower lobe of the right lung (10 mm in the previous examination) and 15 mm in the superior segment of the lower lobe of the left lung (12 mm in the previous examination) are observed. There is an intrapulmonary metastatic lesion measuring 21mm in diameter (19mm in his previous examination) in the lingula inferior segment of the left lung. In his current examination, unlike his previous examination, intralobular septal thickenings consistent with pulmonary fibrosis, traction bronchiectasis and fibrotic ground glass densities are observed in the left lung upper lobe anterior posterior segments, lower lobe superior segment and upper lobe lingula inferior segment. Compatible with pulmonary fibrosis. In the current examination, unlike the previous examination, the effusion between the pleural leaves in the lower lobe of the right lung is quite regressed. In the current examination, the posterobasal segment is between the leaves of the pleura. It measured 2cm in diameter (5.6cm in diameter at its widest point in his previous review). Unlike the current examination, there are mucous plugs within the dilated bronchial lumens in the posterobasal and mediobasal segments of the right lung lower lobe, and in the left lung lower lobe basal segments, and budding tree views in the defined segments are compatible with bronchiolitis. Infectious bronchiolitis was considered compatible with bronchopneumonic infiltration. Pulmonary metastatic lesion, which was observed in the basal segment of the left lung lower lobe in the previous examination, could not be differentiated in the current examination due to the presence of motion artifact in this localization and subsegmental atelectasis in the posterobasal segment. Calibrations of mediastinal main vascular structures were followed naturally. The right main pulmonary artery is lost in the right hilar mass. There is a pericardial effusion reaching 8 mm in the apex of the left ventricle at its widest point among the pericardial leaves. It is present in its previous review and has a slightly regressed appearance. In the upper abdominal organs, including sections; The gallbladder is slightly distended. Measured 39mm in diameter. Pericholecystic smear-like effusion is observed (acute cholecystitis? Clinical correlation is recommended). Density difference is observed in the distal gallbladder corpus (gall sludge?). It is recommended to be evaluated with USG. Cortical cysts with a diameter of 2.7 cm in the left kidney and 1.7 cm in the right kidney were observed. There are nodular lesions (<10HU) evaluated in favor of adenoma with a diameter of 18 mm in the right adrenal gland corpus superior and 12 mm in the left adrenal gland corpus. In the previous examination, the lesion in the L4 vertebral corpus was not included in the current examination.\nImpression: A significant previous examination in terms of size and extent of a mass lesion with an infiltrating course, invading the mediastinum in the right lung hilum, infiltrating the supraclavicular fossa superiorly, the right thyroid lobe and the right brachial plexus truncus with undetectable borders in the esophagus posteriorly, and extending to the intermediate bronchial bifurcation in the inferior in a patient followed up due to lung Ca.  Findings consistent with pulmonary fibrosis in the upper lobe of the left lung and in the superior segment of the lower lobe. The prominent effusion observed in the right pleural leaves in his previous examination is regressed in the current examination. It is observed as a mild pleural effusion in the right lung lower lobe posterior segment pleura.  Traction bronchiectasis in the apical segment of the upper lobe of the right lung and increases in pleuroparachymal fibrotic density are observed more clearly in the current examination.  As a new finding in the current examination, mucus plugs, budding tree sights infectious bronchiolitis, bronchopneumonic infiltrates are a new finding.  Distension in the gallbladder, mild free fluid in the form of pericholecystic smearing, and density change in the vicinity of the sac infundibulum (acute cholecystitis? Clinical correlation and evaluation with USG is recommended).  Cortical cysts in both kidneys.  Nodular lesions consistent with adenoma in both adrenal glands.'],
            padding=False,
            truncation=False,
            return_tensors=None
        )['input_ids'][0]) # 1353



In [91]:
train[train["len"] >= 6000]

,sample_name,report_metadata,subject_id,sentences,labels,volume_label,key_phrase,len
8257,train_3509_a_1,Findings: Mediastinal structures cannot be eva...,train_3509_a,"[Prior left upper lobectomy, Complete loss of ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, ...","[Prior left upper lobectomy, Complete loss of ...",7045
8258,train_3509_a_2,Findings: Mediastinal structures cannot be eva...,train_3509_a,"[Prior left upper lobectomy, Complete loss of ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, ...","[Prior left upper lobectomy, Complete loss of ...",7045
9953,train_4253_a_1,Findings: There is a mass lesion originating f...,train_4253_a,[Mass lesion originating from the right main b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, ...",[Mass lesion originating from the right main b...,6160
24571,train_10563_a_1,Findings: In the axilla and supraclavicular fo...,train_10563_a,[Emphysema is prominent in the upper lobes of ...,"[[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, ...",[Emphysema is prominent in the upper lobes of ...,6037
24572,train_10563_a_2,Findings: In the axilla and supraclavicular fo...,train_10563_a,[Emphysema is prominent in the upper lobes of ...,"[[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, ...",[Emphysema is prominent in the upper lobes of ...,6037


In [92]:
train[train["len"] >= 6000]['report_metadata'].iloc[2]

'Findings: There is a mass lesion originating from the right main bronchus in the right lung hilum, invading the mediastinum superiorly, adjacent to the right trachea, and infiltratively extending to the right anterior scalene muscles. Invasion of the anterior scalene muscle and brachial plexus trunks is observed in the supraclavicular fossa on the right. The first rib extends to the vertebral junction. It extends anteriorly to the strep muscles in the right paratracheal area. Its borders with the right thyroid lobe anteriorly and the esophagus posteromedially cannot be distinguished. It is doubtful in favor of invasion. Since contrast material was not given, its relation with vascular structures was not evaluated. It extends to the left CCA posterior in the previous examination, and it extends to a similar localization in the current examination. Inferiorly, it continues in the mediastinum until distal to the right intermediate bronchus. Pathological LAP with calcification of approxim

In [ ]:
"Findings: Mediastinal structures cannot be evaluated optimally because contrast material is not given. As far as can be followed: It was learned from the patient's history that he had undergone left upper lobectomy. Total loss of aeration is observed in the lower lobe of the left lung, and there is an appearance of consolidation-soft tissue density in the left pulmonary hilus, whose borders cannot be clearly distinguished. When the previous examinations of the patient are examined, increased FDG uptake is observed in PET-CT in this localization. In addition, CT examination reveals that this appearance narrows the pulmonary artery. When evaluated together with these findings, the described appearance was thought to be a mass. It is observed that the described lesion extends along the mediastinal pleura, especially in the medial and anterior sections, and there are pleural thickenings in the left hemithorax. Pleural effusion is observed on the left. Hyperdense appearances are observed in the effusion and it is thought to be due to pleurodesis. In addition, there are thin septum-like appearances in places within the effusion. Minimal pleural effusion is also observed on the right, and there is a drainage catheter in the effusion at the level of the lower lobe of the lung. A mass whose borders cannot be distinguished from the pleura is observed in the left hemithorax at the level where the 2nd and 3rd ribs articulate with the sternum. The anterior-posterior diameter of the described mass was approximately 35 mm at its thickest point. Just caudal to this, another soft tissue lesion measuring approximately 20 mm in diameter is observed in the subcutaneous adipose tissue, adjacent to the 5th rib and sternum joint, and it was evaluated in favor of metastasis. There is a mass with the longest diameter of 54 mm in the subcutaneous adipose tissue in the lateral part of the left hemithorax. In addition, there is another similar solid mass with a diameter of 54 mm on the posterolateral at the lower part of the left hemithorax. Those masses were evaluated in favor of metastases. In the left hemithorax, at the level of the middle part of the hemithorax, infiltrative appearances are observed in the intercostal area and subcutaneous adipose tissue, in soft tissue density whose borders cannot be distinguished from the muscle groups and pleura in this localization. Although these appearances are nonspecific in this examination, when evaluated together with the patient's previous examinations, these appearances were thought to be neoplastic infiltration. Heart contour and size are normal. There is minimal pericardial effusion. The widths of the mediastinal main vascular structures are normal. There are lymphadenopathies in the mediastinum and hilar regions. In addition, nodular lesions are observed in the paracardiac fat pad and were evaluated in favor of lymphadenopathies. The largest of the described lymphadenopathies is observed in the right hilar region and its short diameter is 22 mm. There is no pathological wall thickness increase in the esophagus within the sections. There is no obstructive pathology in the trachea and right main bronchus. There are emphysematous changes and occasional pleuroparenchymal sequelae in the right lung. In the apical segment of the upper lobe of the right lung, an irregularly bordered soft tissue density with a longest diameter of 30 mm is observed. When this appearance was evaluated together with previous examinations, it was thought to be a metastatic mass. The mass boundaries cannot be distinguished from the pleura. In addition, there are masses in the right lung adjacent to the upper and lower lobes, which are thought to originate primarily from the pleura and most of them are in the form of plaques. The largest of these masses are observed within the horizontal and oblique fissures and measure approximately 29 mm in the thickest part (series 2 section 173). These appearances were evaluated in favor of metastases. There are also millimetric nodules in the right lung. No infiltrative lesion was detected in the right lung. Hypodense lesions are observed in the liver. These lesions can be observed in both lobes. When evaluated together with previous examinations, it was understood that these appearances were metastases. The larger metastatic lesions are observed in segment 4A and segment 3, and their longest diameters were 48 mm and 41 mm at their widest points, respectively. There is a nodular lesion on the lateral leg of the left adrenal gland, measuring approximately 20 mm in longest diameter. Again, another nodular lesion measuring approximately 7 mm in diameter is observed in the vicinity of the left adrenal gland corpus. The appearances described in the presence of primary disease were primarily evaluated in favor of metastases. Thickening is also observed in the medial leg of the right adrenal gland. The described appearance is also present in the previous examination of the patient. However, it was found to be enlarged on this examination. Therefore, it was thought to be metastasis. No upper abdominal free fluid was detected in the sections. There are lymphadenopathies in the interaorthocaval and paracaval regions. The largest of the described lymphadenopathies is observed in the paracaval area and its short diameter is 15 mm. No appearance that can be evaluated in favor of metastasis was detected in the bone structures within the sections. There was no significant difference in the size of lymphadenopathy in the mediastinum and in the hilar region. No significant difference was found in the size of the mass in the infiltrative character described in the left hemithorax. It is understood that the masses observed in the subcutaneous adipose tissue in the left hemithorax have grown significantly. No significant difference was found in the sizes of the masses with infiltrative character described in the intercostal region. It is understood that the dimensions of the mass described in the upper lobe apical segment of the right lung have increased slightly. In addition, there is a slight increase in the size of the masses described in the right lung and thought to be of pleural origin. There is an increase in the size of metastatic lesions observed in the liver. An increase in the size of both masses adjacent to the left adrenal gland was observed. There is also an increase in lesion size in the right adrenal gland.\nImpression: Operated lung Ca, left upper lobectomy, total aeration loss in the left lung and infiltrative appearance in the left pulmonary hilum, which is thought to be a mass when evaluated together with the patient's previous examinations, masses compatible with metastasis-infiltration in the subcutaneous fat tissue and muscle groups in the left hemithorax, metastatic in the right lung masses, lymphadenopathies in the mediastinal and hilar region, liver metastases, metastases in both adrenal glands, metastases adjacent to the left adrenal gland, intraabdominal lymphadenopathies."

In [97]:
total_sen_list = []
for i in train["sentences"].values:
    total_sen_list += i

In [99]:
t = pd.DataFrame()

t['sen'] = total_sen_list
t['len'] = t['sen'].map(lambda x: len(x))

In [110]:
t[t['len'] >= 380]

,sen,len
231310,Centriacinar nodular densities with a tree-in-...,388
231322,Centriacinar nodular densities with a tree-in-...,388
371464,Nonspecific pulmonary nodules: subpleural nodu...,382
371482,Nonspecific pulmonary nodules: subpleural nodu...,382


In [109]:
t[t['len'] >= 380].iloc[-1].values[0]

'Nonspecific pulmonary nodules: subpleural nodule in the laterobasal segment of the right lower lobe, subpleural nodule in the laterobasal segment of the left lower lobe, fissure-level nodule in the apicoposterior segment of the left upper lobe, nodule in the inferior lingular segment, nodule in the left lingular segment, and nodule in the posterior segment of the right upper lobe'

In [ ]:
len(self.tokenizer(
            ['Centriacinar nodular densities with a tree-in-bud appearance in the posterior segment of the right upper lobe, the superior segment of the right lower lobe, the medial segment of the right middle lobe, the anterior segment of the left upper lobe, the lingular segment of the left upper lobe, the superior segment of the left lower lobe, and the posterobasal segment of the left lower lobe'],
            padding=False,
            truncation=False,
            return_tensors=None
        )['input_ids'][0]) # 90

In [ ]:
len(self.tokenizer(
            ['Nonspecific pulmonary nodules: subpleural nodule in the laterobasal segment of the right lower lobe, subpleural nodule in the laterobasal segment of the left lower lobe, fissure-level nodule in the apicoposterior segment of the left upper lobe, nodule in the inferior lingular segment, nodule in the left lingular segment, and nodule in the posterior segment of the right upper lobe'],
            padding=False,
            truncation=False,
            return_tensors=None
        )['input_ids'][0]) # 97

# 불완전 case

In [1]:
import pandas as pd

train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_phrase_align.json", lines=True)
val   = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_phrase_align.json", lines=True)


In [6]:
import glob


target_list = glob.glob("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b_phrase/train_2048/embeddings/*")
# target_list = glob.glob("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b_phrase/val_2048/embeddings/*")

In [7]:
import torch
from tqdm import tqdm

for i in tqdm(target_list):
    t = torch.load(i)
    if len(t.keys()) != 2:
        print(i)

100%|██████████| 46375/46375 [03:38<00:00, 212.46it/s]


In [8]:
target_cache_list = [
    "064f9e8b5764b635fffad011eef10ec064ff2757974108b64ab2e942c2cb49f3",
    "12827bb7cb2dae6a941050716fec8f59844815d8a7d9a28e9d40c1c882113eae",
    "2a65515a75691b6484cd9a44dd94a236f920f0718bd363228ac70249ce46c9fd",
    "2a90af338f7a498c65be8f2dcae262430ebf6811fb7f250907710bf4ee34675e",
    "2dcab63d57a6e72a40fa6b45937b8c5e257a2913ef2c37eaef17b89d29c9eba5",
    "3c284608b9a0d439e2fb2697ea66163d07f2e8d51dff4995a6b8df4640e175bc",
    "3c64f15822615f08fb5c9b51342366cc89351faeb07eb05224c1f7de5cd3743f",
    "40ea58274224e10e05823c42d052630fe64ca0aa27fa22cfa58006cdca57f1f9",
    "48fffdf7f14596c7237c0cb6f504379591fd00375d050d7e5c8fbc9f0b6c57f9",
    "4b278a22cccf6d3953b6e43b3cd75bb70fc340e7aaaeba7f573d508a4f0a8092",
    "4ce936304cc63eb72a48820c1a43ec1f84d0696b76833510600305f5a120758e",
    "5a95dd92f6266526ea84753daad881098bd92173732032c33c3a30189cedcdd2",
    "6029e64a8d02c8c211e631e8d55bca068b47415511feb79e0b9b5dac2b0743da",
    "70e0b0a654cd32ff1f89029616cdb0f91e906df2fefc6ac03a55e51941a051d5",
    "7a191ccbe51d811743653843a3cbdfefe7f35da835e4639f38fbe2c82683fbea",
    "7d22ef77a401ddcc2dd2cfd7190ba14db7f406e1ca053fb4343a80a67d406dd3",
    "7fbe63544efe2fef9693675440e96aee6ac883ce2f305f1fffeb04a585cf6809",
    "82ee5940740184ec2c1777c49cf5c3f70a73ee7f432134f6c5a55da15c16064e",
    "8d1f64c790c144d07823b921d89fa3875d9f9eae5d0336cd63e15e06ea0ded38",
    "9670de92b7128258f18859f0c96fa5de86694f09c140212583eef3e0b9192716",
    "aa641473207f7187bb9d8c7f05f76b5f16363f77d14b6bbbb002aae00e317bbc",
    "abc6758793dc6f852ab44d94e03af8dc2ca4a27572145530e30f2bfe4c1311ec",
    "acf87b27cf96514a884ecde082ac1e79257d071a58f6fc7ba242394ab7b4c65c",
    "b44084abc6ede73b33d0a9c8f38b51b6689e4f9cf8dbdc3bf45c29d30ea4e643",
    "bbb0b0eeda9dd4ff872236fb51f97acb4b1b6cd77a495cddbf31182eceddd97f",
    "bbb22749b6f0c97c02306fcf467578af7168b845ed5eae121559ee3a0c6ebd18",
    "c3fda3b4627b42f8d0786a7ab60515e2e9c7933e439a110f47f89e36e1f48ef9",
    "d9d2d2e5928b8d67f97c05b12128a3b1f35917468075e781fd59010f1af0edf2",
    "dc7cbb5bf4dc19b9258e719140bda09538ba5404238993a0559c70756017610c",
    "df2295549142b5a24c4538f4a09e5f06e446b57e518a6b55cd00c79bbb44cd6a",
    "e2fb25b529fadc83c9be82353345d6849c5cd8919f846b4da9f71b5f1ce65573",
    "ebc8cee29d703a947fbbf9f6b333d08098db7faa3b332e7726aabe0905fc0c18",
    "ec7081609490c8f9d9ddee3bc7149e5b87e1cf06c4f2056c4d3d98dfee3bd0ba",
    "f3235f4a4e98eb470dc10fb92608ae0cce8636918ed8e47936047d08ea6fe9c2",
    "f818860494f7bf35d03d7ecdda68fa4928f2c6c64f9882049ad12d9f010ca018",
]

In [ ]:
[Warning] 캐시 불완전 - sample: train_17047_a_1, missing keys: ['all_findings']
[Warning] 캐시 불완전 - sample: train_10774_a_1, missing keys: ['all_findings']

In [113]:
target = train[train['sample_name'] == "train_17047_a_1"]

In [ ]:
target

In [5]:
import hashlib

train['cache'] = train['sample_name'].map(lambda x: hashlib.sha256(x.encode()).hexdigest())

In [10]:
# train['cache']에서 target_cache_list에 일치하는 case만 살리기
target_df = train[train['cache'].isin(target_cache_list)]

In [11]:
target_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/target.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )


In [ ]:
import hashlib

hashlib.sha256(x.encode()).hexdigest()
    

'0db97e6b9eea1389eb53af886ca87a882a8e23aaae3f1d11751b75f22cb83180'

In [65]:
import torch

a = torch.load("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b_phrase/train/embeddings/0db97e6b9eea1389eb53af886ca87a882a8e23aaae3f1d11751b75f22cb83180.pt")

In [81]:
import torch

matrix = torch.tensor([[ 0.001,  0.001,  0.7], [ 0.001,  0.8,  0.001], [ 0.9,  0.001,  0.001]])
norm = matrix.norm(p=2, dim=-1, keepdim=True)
normalized = matrix / norm.clamp_min(0.0)

In [82]:
normalized

tensor([[0.0014, 0.0014, 1.0000],
        [0.0012, 1.0000, 0.0012],
        [1.0000, 0.0011, 0.0011]])

In [83]:
normalized.shape

torch.Size([3, 3])

In [84]:
normalized @ normalized.T

tensor([[1.0000, 0.0027, 0.0025],
        [0.0027, 1.0000, 0.0024],
        [0.0025, 0.0024, 1.0000]])

In [12]:
import torch

a = torch.load("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/text_cache_qwen3_embedding_8b/valid_test_debug/embeddings/4ce936304cc63eb72a48820c1a43ec1f84d0696b76833510600305f5a120758e.pt")

In [13]:
a.keys()

dict_keys(['all_findings', 'key_phrase'])

In [117]:
a['key_phrase']

tensor([[ 0.0112,  0.0189,  0.0131,  ..., -0.0012, -0.0010,  0.0112],
        [ 0.0122,  0.0061,  0.0106,  ...,  0.0039,  0.0109,  0.0058],
        [ 0.0295,  0.0035, -0.0067,  ...,  0.0049, -0.0156,  0.0057],
        ...,
        [-0.0011,  0.0064,  0.0137,  ...,  0.0009, -0.0056,  0.0026],
        [ 0.0183, -0.0035, -0.0090,  ...,  0.0021, -0.0127, -0.0010],
        [ 0.0247,  0.0165, -0.0135,  ...,  0.0071, -0.0091,  0.0041]])

In [ ]:
Atelectasis : "Atheromatous plaques in the aorta and coronary arteries",

Pleural effusion": "Mediastinal structures cannot be optimally evaluated due to the lack of contrast material",

# Reduce cases

In [3]:
import pandas as pd


train_df = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob.json", lines=True)
valid_df = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_prob.json", lines=True)

In [4]:
train_df

,sample_name,report_metadata,subject_id,key_phrase,sentences,labels,volume_label,sentence_probs
0,train_1_a_1,Findings: Multiple venous collaterals are pres...,train_1_a,"[Patent trachea and both main bronchi, Linear ...","[Patent trachea and both main bronchi, Linear ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
1,train_1_a_2,Findings: Multiple venous collaterals are pres...,train_1_a,"[Patent trachea and both main bronchi, Linear ...","[Patent trachea and both main bronchi, Linear ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
2,train_2_a_1,Findings: Trachea and both main bronchi were i...,train_2_a,[Centrally positioned and patent trachea and b...,[Centrally positioned and patent trachea and b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, ...","[[0.0024000000000000002, 0.0001, 0.0002, 0.000..."
3,train_2_a_2,Findings: Trachea and both main bronchi were i...,train_2_a,[Centrally positioned and patent trachea and b...,[Centrally positioned and patent trachea and b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, ...","[[0.0024000000000000002, 0.0001, 0.0002, 0.000..."
4,train_3_a_1,Findings: Right thyroid lobe sizes increased. ...,train_3_a,"[Minimal bronchiectatic changes, Peribronchial...","[Minimal bronchiectatic changes, Peribronchial...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, ...","[[0.058600000000000006, 0.01, 0.0154, 0.017400..."
...,...,...,...,...,...,...,...,...
46370,train_19998_a_2,Findings: Trachea and lumen of both main bronc...,train_19998_a,"[Patent trachea and both main bronchi, Branchi...","[Patent trachea and both main bronchi, Branchi...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
46371,train_19999_a_1,"Findings: Trachea, both main bronchi are open....",train_19999_a,"[Patent trachea and both main bronchi, Diffuse...","[Patent trachea and both main bronchi, Diffuse...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
46372,train_19999_a_2,"Findings: Trachea, both main bronchi are open....",train_19999_a,"[Patent trachea and both main bronchi, Diffuse...","[Patent trachea and both main bronchi, Diffuse...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
46373,train_20000_a_1,Findings: CTO is within normal limits. Thymic ...,train_20000_a,[Trachea and main bronchi are normal in calibe...,[Trachea and main bronchi are normal in calibe...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, ...","[[0.0009000000000000001, 0.0001, 0.0002, 0.000..."


In [5]:
train_df = train_df[train_df["sample_name"].str.endswith("_1", na=False)].copy()
train_df = train_df.drop_duplicates(subset=["subject_id"], keep="first")

In [6]:
valid_df = valid_df[valid_df["sample_name"].str.endswith("_1", na=False)].copy()
valid_df = valid_df.drop_duplicates(subset=["subject_id"], keep="first")

In [7]:
train_df

,sample_name,report_metadata,subject_id,key_phrase,sentences,labels,volume_label,sentence_probs
0,train_1_a_1,Findings: Multiple venous collaterals are pres...,train_1_a,"[Patent trachea and both main bronchi, Linear ...","[Patent trachea and both main bronchi, Linear ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
2,train_2_a_1,Findings: Trachea and both main bronchi were i...,train_2_a,[Centrally positioned and patent trachea and b...,[Centrally positioned and patent trachea and b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, ...","[[0.0024000000000000002, 0.0001, 0.0002, 0.000..."
4,train_3_a_1,Findings: Right thyroid lobe sizes increased. ...,train_3_a,"[Minimal bronchiectatic changes, Peribronchial...","[Minimal bronchiectatic changes, Peribronchial...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, ...","[[0.058600000000000006, 0.01, 0.0154, 0.017400..."
6,train_3_b_1,Findings: Central venous catheter is seen on t...,train_3_b,[Patent trachea and both main bronchi without ...,[Patent trachea and both main bronchi without ...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, ...","[[0.0012000000000000001, 0.0002, 0.00030000000..."
8,train_4_a_1,Findings: No occlusive pathology was observed ...,train_4_a,[Patent trachea and both main bronchi without ...,[Patent trachea and both main bronchi without ...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, ...","[[0.0012000000000000001, 0.0002, 0.00030000000..."
...,...,...,...,...,...,...,...,...
46365,train_19996_a_1,Findings: Trachea and both main bronchi were i...,train_19996_a,[Centrally positioned and patent trachea and b...,[Centrally positioned and patent trachea and b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[[0.0024000000000000002, 0.0001, 0.0002, 0.000..."
46367,train_19997_a_1,Findings: Cardiothoracic ratio increased in fa...,train_19997_a,[Linear subsegmental atelectasis in both lungs...,[Linear subsegmental atelectasis in both lungs...,"[[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, ...","[[0.0151, 0.0012000000000000001, 0.0083, 0.011..."
46369,train_19998_a_1,Findings: Trachea and lumen of both main bronc...,train_19998_a,"[Patent trachea and both main bronchi, Branchi...","[Patent trachea and both main bronchi, Branchi...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
46371,train_19999_a_1,"Findings: Trachea, both main bronchi are open....",train_19999_a,"[Patent trachea and both main bronchi, Diffuse...","[Patent trachea and both main bronchi, Diffuse...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."


In [8]:
valid_df

,sample_name,report_metadata,subject_id,key_phrase,sentences,labels,volume_label,sentence_probs
0,valid_1_a_1,"Findings: Trachea, both main bronchi are open....",valid_1_a,"[Patent trachea and both main bronchi, A few s...","[Patent trachea and both main bronchi, A few s...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
2,valid_2_a_1,Findings: As far as can be seen; A stable soft...,valid_2_a,[Contour irregularities in the pleura of the l...,[Contour irregularities in the pleura of the l...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, ...","[[0.00030000000000000003, 0.0002, 0.0002, 0.00..."
4,valid_3_a_1,Findings: Trachea was in the midline of both m...,valid_3_a,[Centrally positioned and patent trachea and b...,[Centrally positioned and patent trachea and b...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...","[[0.0024000000000000002, 0.0001, 0.0002, 0.000..."
6,valid_4_a_1,Findings: There is minimal pleural effusion on...,valid_4_a,"[Minimal right pleural effusion, No left pleur...","[Minimal right pleural effusion, No left pleur...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, ...","[[0.0137, 0.0024000000000000002, 0.05690000000..."
8,valid_4_b_1,"Findings: Trachea, both main bronchi are open....",valid_4_b,"[Patent trachea and both main bronchi, Right p...","[Patent trachea and both main bronchi, Right p...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[[0.0112, 0.0001, 0.00030000000000000003, 0.00..."
...,...,...,...,...,...,...,...,...
2992,valid_1300_a_1,Findings: Trachea and both main bronchi are no...,valid_1300_a,[Normal and patent trachea and both main bronc...,[Normal and patent trachea and both main bronc...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0.0011, 0.0001, 0.0002, 0.0004, 0.0005, 0.00..."
2994,valid_1301_a_1,Findings: Trachea and both main bronchi were o...,valid_1301_a,[Patent trachea and both main bronchi without ...,[Patent trachea and both main bronchi without ...,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0.0012000000000000001, 0.0002, 0.00030000000..."
2996,valid_1302_a_1,Findings: Trachea and main bronchi are open. R...,valid_1302_a,"[Patent trachea and main bronchi, Peripheral a...","[Patent trachea and main bronchi, Peripheral a...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...","[[0.0089, 0.0001, 0.00030000000000000003, 0.00..."
2998,valid_1303_a_1,Findings: Evaluation of solid organs and major...,valid_1303_a,"[Midline trachea and patent both main bronchi,...","[Midline trachea and patent both main bronchi,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, ...","[[0.0013000000000000002, 0.0001, 0.0002, 0.000..."


In [9]:
train_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob_half.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

valid_df.to_json(
        "/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_valid_filtered_prob_half.json", 
        orient="records", 
        lines=True,           # 가독성을 위해 들여쓰기 4칸 추가
        force_ascii=False     # 한글/특수문자 깨짐 방지
    )

# ZeroShot 체크

In [1]:
import pandas as pd

train = pd.read_json("/data/drdcad/sunggu/0.Challenge/pillar-pretrain/data/merged_train_filtered_prob_half.json", lines=True)



In [ ]:
    # ------------------------------------------------------------------ #
    CTRATE_POS_WEIGHT = [
        7.10,   # Medical material                (prev 12.34%)
        2.52,   # Arterial wall calcification     (prev 28.37%)
        7.88,   # Cardiomegaly                    (prev 11.26%)
        12.82,  # Pericardial effusion            (prev  7.24%)
        2.92,   # Coronary artery wall calcif.    (prev 25.50%)
        5.98,   # Hiatal hernia                   (prev 14.32%)
        2.86,   # Lymphadenopathy                 (prev 25.92%)
        4.17,   # Emphysema                       (prev 19.35%)
        2.84,   # Atelectasis                     (prev 26.01%)
        1.21,   # Lung nodule                     (prev 45.35%)
        1.71,   # Lung opacity                    (prev 36.95%)
        2.75,   # Pulmonary fibrotic sequela      (prev 26.70%)
        7.26,   # Pleural effusion                (prev 12.10%)
        11.96,  # Mosaic attenuation pattern      (prev  7.72%)
        8.48,   # Peribronchial thickening        (prev 10.55%)
        4.67,   # Consolidation                   (prev 17.64%)
        8.96,   # Bronchiectasis                  (prev 10.04%)
        11.59,  # Interlobular septal thickening  (prev  7.94%)
    ]

In [15]:
for i in train['key_phrase'].values:
    for j in i:
        if "there" in j or "There" in j:
            print(j)

        if "effusion" in j:
            print(j)


No pericardial effusion or thickening
No pericardial effusion or thickening
No pericardial effusion or pericardial thickening
Bilateral pleural effusions extending to the apices of both lungs in the supine position
Right pleural effusion
No pericardial effusion
No pericardial effusion or thickening
No pericardial effusion or thickening
No pleural effusion
No pericardial effusion
No bilateral pleural thickening or effusion
No pericardial effusion or thickening
Bilateral pleural effusions with associated compressive atelectasis
Minimal pericardial effusion
No pleural effusion, pneumothorax, or pneumonia
No pericardial effusion or thickening
No pericardial effusion or thickening
No pericardial effusion or thickening
No pericardial effusion or thickening
No pleural effusion or pleural thickening
No pericardial effusion or thickening
No pleural effusion or pleural thickening
No pericardial effusion or thickening
No pericardial effusion or thickening
pleural effusion in both hemithoraces
No 

In [8]:
len(train['key_phrase'].values)

23869

In [9]:
train['key_phrase'].values[0]

['Patent trachea and both main bronchi',
 'Linear atelectasis in both lungs',
 'Subsegmental atelectasis in the right middle lobe',
 'Bronchial wall thickening in the bilateral lower lobes',
 'Peribronchial tree-like reticulonodular densities in the bilateral lower lobes',
 'Focal minimal peribronchial consolidation in the lower lobes',
 'Calcific plaques in the aortic arch',
 'Normal mediastinal vascular structures, heart contour, and size',
 'Normal thoracic aorta diameter',
 'No pericardial effusion or thickening',
 'Collapsed left subclavian vein',
 'Normal thoracic esophagus caliber without significant wall thickening',
 'No enlarged lymph nodes in the prevascular, paratracheal, subcarinal, or bilateral hilar-axillary regions',
 'Multiple venous collaterals in the anterior left chest wall associated with the anterior jugular vein at the level of the right sternoclavicular junction',
 'Osteophytes with anterior extension in the thoracic vertebrae',
 'Atrophic left kidney',
 'Right 